# Google Gemini u pomoći kod SNA-a (mock mreža + interpretacija)

**Poglavlje:** [15. Generativni AI (Gemini) u SNA-u](../content/15_ai_gemini_u_sna.md)

Cilj: pokazati **siguran** tijek — **NetworkX** izračuna jednostavne mjere na **mock** grafu, a **Gemini** (ako je postavljen `GEMINI_API_KEY` u `.env`) može predložiti *nacrt* kulturne interpretacije. Bez ključa bilježnica i dalje radi (ispisuje upute).

## Besplatan API ključ: Google AI Studio

1. Otvorite **[Google AI Studio](https://aistudio.google.com/)**.
2. Prijavite se Google računom.
3. Odaberite **Get API key** → **Create API key** (novi ili postojeći Cloud projekt).
4. Kopirajte ključ u datoteku **`.env`** u **korijenu** ovog repozitorija (jedna linija, bez navodnika oko vrijednosti):

   ```env
   GEMINI_API_KEY=vaš_dugi_ključ_ovdje
   ```

5. Pokrenite Jupyter iz korijena projekta (`IstrazivanjeDrustvenihMreza`) ili pustite ćeliju ispod koja traži `.env` u trenutnoj mapi i roditeljima.
6. **Nikad** ne commitajte `.env` u git. Ako ključ procuri, opozovite ga u konzoli i napravite novi.

Besplatna razina ima **limite** zahtjeva; detalji su na [ai.google.dev](https://ai.google.dev/pricing).

## Ovisnosti

U terminalu (aktiviran `venv`):

```bash
pip install google-generativeai python-dotenv
```

(Već su navedene u `requirements.txt` repozitorija.)

In [ ]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None


def load_env_from_repo() -> None:
    if load_dotenv is None:
        return
    start = Path.cwd().resolve()
    for folder in [start, *start.parents][:5]:
        env_file = folder / ".env"
        if env_file.is_file():
            load_dotenv(env_file)
            print("Učitano:", env_file)
            return
    print("Nema .env u trenutnoj mapi ni nekoliko roditelja — postavite GEMINI_API_KEY u okruženju ručno.")


load_env_from_repo()

API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
MODEL_ID = os.environ.get("GEMINI_MODEL", "gemini-2.0-flash")
print("Ključ postavljen:", "da" if API_KEY else "ne (samo lokalni ispis bez API-ja)")
print("Model:", MODEL_ID)

## Mock mreža i mjere (NetworkX)

Isti princip kao u poglavlju 10 — **sintetički** čvorovi, bez stvarnih korisnika.

In [ ]:
import json

import networkx as nx

G = nx.Graph()
G.add_edges_from(
    [
        ("festival", "lokalni_portal"),
        ("festival", "volonter"),
        ("lokalni_portal", "kriticar"),
        ("kriticar", "nacionalni_medij"),
        ("nacionalni_medij", "muzej"),
    ]
)

summary = {
    "nodes": list(G.nodes()),
    "edges": list(G.edges()),
    "degree_centrality": nx.degree_centrality(G),
    "betweenness_centrality": nx.betweenness_centrality(G),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Gemini: nacrt interpretacije (samo mock podaci u promptu)

U prompt šaljemo **isključivo** gornji JSON (bez imena osoba, bez tweetova). U radu s pravim podacima najprije **etika**, **anonimizacija** i odobrenje ustanove.

In [ ]:
PROMPT = f"""Ti si tutor na kolegiju o društvenim mrežama (kulturalne studije).
Ispod je JSON s MOCK podacima o maloj nesmjerenoj mreži (čvorovi su tipovi aktera, ne ljudi).
Zadatak: napiši kratku interpretaciju (5–8 rečenica na hrvatskom) koja:
- povezuje degree i betweenness s mogućim ulogama u javnosti oko kulturnog događaja;
- izričito kaže da su podaci simulirani i da nema kauzalnih zaključaka;
- predloži jedan smislen sljedeći korak u istraživanju (npr. kvalitativno čitanje sadržaja).

Podaci:
{json.dumps(summary, ensure_ascii=False)}
"""

if not API_KEY:
    print("Preskačem API: dodajte GEMINI_API_KEY u .env i ponovno pokrenite kernel.")
else:
    try:
        import google.generativeai as genai
    except ImportError:
        print("Instalirajte: pip install google-generativeai")
    else:
        genai.configure(api_key=API_KEY)
        model = genai.GenerativeModel(MODEL_ID)
        try:
            resp = model.generate_content(PROMPT)
            print(resp.text)
        except Exception as e:
            print("Greška API-ja (kvota, model ili mreža):", e)
            print("Probajte u .env postaviti GEMINI_MODEL=gemini-1.5-flash")

## Zaključak

- AI je **alat za nacrte i ubrzanje**, ne autor znanstvenog zaključka.
- Uvijek **citirajte** ljudsku i sekundarnu literaturu; **ne** citirajte model kao izvor činjenica.
- Za stvarne podatke s platformi: **GDPR**, ToS, etičko povjerenstvo — vidi poglavlja **10**, **13** i **14**.